# Comparison: Old CSV vs New Excel Catalogue

This notebook:
1. Loads the old file (`Cataloguing screening.csv`) and the new file (`IMPULSE_WP3-T3.4_catalogue_08072025_v2.xlsx`)
2. Compares their structures (columns, value counts, overlaps)
3. Creates a new csv that maps the new data into the old column format so it can be used as a drop-in replacement in the dashboard

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. Load both datasets

In [2]:
old_df = pd.read_csv('Cataloguing screening.csv')
new_df = pd.read_excel(
    'IMPULSE_WP3-T3.4_catalogue_08072025_v2.xlsx',
    sheet_name='Catalogue_Models (251216)',
)
dropdown_df = pd.read_excel(
    'IMPULSE_WP3-T3.4_catalogue_08072025_v2.xlsx',
    sheet_name='drop down menu',
)

print(f'Old CSV shape: {old_df.shape}')
print(f'New Excel shape: {new_df.shape}')

Old CSV shape: (57, 13)
New Excel shape: (56, 26)


## 2. Column comparison

In [3]:
print('=== OLD CSV Columns ===')
for i, c in enumerate(old_df.columns, 1):
    print(f'  {i:2d}. {c}')

print(f'\n=== NEW Excel Columns ===')
for i, c in enumerate(new_df.columns, 1):
    print(f'  {i:2d}. {c}')

=== OLD CSV Columns ===
   1. Site
   2. Disease Area
   3. Type
   4. Publication information
   5. Model organism
   6. Cell type
   7. Cell Type Specifics
   8. Assay format
   9. Assay readout
  10. Additional assay information
  11. # of compounds
  12. Single Point or Dose Response
  13. Additional Notes

=== NEW Excel Columns ===
   1. Partner Site Name
   2. Partner Site Country
   3. Primary Disease Area
   4. Tissue of origin (free text)
   5. Disease/indication (free text)
   6. Type
   7. Model Organism
   8. Biological System
   9. Co-culture?
  10. Model and modification details (free text)
  11. Cell line name(s) (free text)
  12. Plate density
  13. Assay Design (1)
  14. Assay Design (2)
  15. if 3D: assay type
  16. Assay readout
  17. Assay readout (2)
  18. Assay readout (3)
  19. Additional assay information (free text)
  20. Data Points
  21. Screening Mode
  22. Measurement Type
  23. Additional Details (timepoint, points DRC) 
(free text)
  24. Publication Detai

## 3. Column mapping (Old → New)

| Old Column | New Column | Notes |
|---|---|---|
| Site | Partner Site Name | Renamed |
| Disease Area | Primary Disease Area | Renamed |
| Type | Type | Same |
| Publication information | Publication Details (doi only) | Renamed |
| Model organism | Model Organism | Case change |
| Cell type | Biological System | Different semantics — old had specific terms (co-culture, organoid, etc.), new has categories |
| Cell Type Specifics | Model and modification details (free text) | Closest match |
| Assay format | Assay Design (1) + Assay Design (2) | Need to merge |
| Assay readout | Assay readout + (2) + (3) | New has 3 readout cols |
| Additional assay information | Additional assay information (free text) | Renamed |
| # of compounds | Data Points | Different format (counts → ranges) |
| Single Point or Dose Response | Screening Mode | Renamed + different values |
| Additional Notes | Additional Details (timepoint, points DRC) | Renamed |

**New columns not in old:** Partner Site Country, Tissue of origin, Disease/indication, Co-culture?, Cell line name(s), Plate density, if 3D: assay type, Assay readout (2)/(3), Measurement Type, Available to the User, Comments regarding the availability

## 4. Side-by-side unique value comparison

In [4]:
# Mapping of comparable columns: (old_col, new_col)
comparable = [
    ('Site', 'Partner Site Name'),
    ('Disease Area', 'Primary Disease Area'),
    ('Type', 'Type'),
    ('Model organism', 'Model Organism'),
    ('Cell type', 'Biological System'),
    ('Assay format', 'Assay Design (1)'),
    ('Assay readout', 'Assay readout'),
    ('# of compounds', 'Data Points'),
    ('Single Point or Dose Response', 'Screening Mode'),
]

for old_col, new_col in comparable:
    old_vals = set(old_df[old_col].dropna().str.strip()) if old_df[old_col].dtype == 'object' else set(old_df[old_col].dropna())
    new_vals = set(new_df[new_col].dropna().str.strip()) if new_df[new_col].dtype == 'object' else set(new_df[new_col].dropna())
    print(f'\n--- {old_col} (old) vs {new_col} (new) ---')
    print(f'  Old unique ({len(old_vals)}): {sorted(old_vals)}')
    print(f'  New unique ({len(new_vals)}): {sorted(new_vals)}')
    overlap = old_vals & new_vals
    if overlap:
        print(f'  Overlap: {sorted(overlap)}')
    only_old = old_vals - new_vals
    only_new = new_vals - old_vals
    if only_old:
        print(f'  Only in old: {sorted(only_old)}')
    if only_new:
        print(f'  Only in new: {sorted(only_new)}')


--- Site (old) vs Partner Site Name (new) ---
  Old unique (11): ['CBCS–KI', 'CIPF', 'Fraunhofer ITMP', 'HZI', 'IBCH PAS', 'MEDINA', 'UC', 'UH-FIMM', 'USC', 'UTU', 'i3S']
  New unique (10): ['CBCS-KI', 'CIPF', 'Fraunhofer ITMP', 'HZI', 'IBCH PAS', 'MEDINA', 'UH-FIMM', 'USC', 'UTU', 'i3S']
  Overlap: ['CIPF', 'Fraunhofer ITMP', 'HZI', 'IBCH PAS', 'MEDINA', 'UH-FIMM', 'USC', 'UTU', 'i3S']
  Only in old: ['CBCS–KI', 'UC']
  Only in new: ['CBCS-KI']

--- Disease Area (old) vs Primary Disease Area (new) ---
  Old unique (2): ['bacterial and viral infection', 'to be completed']
  New unique (5): ['Cancer', 'Hematology', 'Infectious', 'Metabolic', 'Neurological']
  Only in old: ['bacterial and viral infection', 'to be completed']
  Only in new: ['Cancer', 'Hematology', 'Infectious', 'Metabolic', 'Neurological']

--- Type (old) vs Type (new) ---
  Old unique (4): ['Disease Profiling', 'Hit follow-up', 'Primary screening', 'Toxicity models']
  New unique (4): ['Disease profiling', 'Hit follow-

## 5. Showing the drop-down menu sheet


In [ ]:
print('Drop-down menu sheet columns:')
for i, c in enumerate(dropdown_df.columns, 1):
    print(f'  {i:2d}. {c}')

print('\nFirst few rows (row 0 = updated names, row 1 = new column headers):')
dropdown_df.head(3)

Drop-down menu sheet columns:
   1. Original Names
   2. Country of the PS
   3. Partner site number
   4. Unnamed: 3
   5. Partner site name
   6. Disease Area
   7. Unnamed: 6
   8. Type
   9. Publication information
  10. Model organism
  11. Cell type
  12. in BAO co-culture is in assay Design
  13. Unnamed: 12
  14. Unnamed: 13
  15. Plate density
  16. Assay format
  17. Unnamed: 16
  18. Assay readout
  19. Additional assay information
  20. # of compounds
  21. Single Point, Dose Response, Combination
  22. Unnamed: 21
  23. Unnamed: 22
  24. Unnamed: 23
  25. Unnamed: 24

First few rows (row 0 = updated names, row 1 = new column headers):


,Original Names,Country of the PS,Partner site number,Unnamed: 3,Partner site name,Disease Area,Unnamed: 6,Type,Publication information,Model organism,...,Assay format,Unnamed: 16,Assay readout,Additional assay information,# of compounds,"Single Point, Dose Response, Combination",Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,Updated Names,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Partner Site Name,Country of the PS,Service Name,Partner site name,Disease Area,Tissue of origin (free text),Type,Publication information (doi only),Model organism,...,Assay Design,3D assay type,Assay readout,Additional assay information (free text),Data Points,Screening Mode,Measurement Type,"Additional Details (timepoint, points DRC) \n(...",Available to the User,Comments regarding the availability
2,NaN,IMG,CZ,"Advanced cellular model for screening, functio...",NaN,Cancer,"eg. Lung cancer, human hepatocytes",Primary screening,NaN,Human,...,2D,Spheroid/Scaffold-free,Luminescence,NaN,<100,Single Point,Kinetic,NaN,Yes,NaN


In [ ]:

dd_cols_of_interest = {
    'Disease Area': dropdown_df['Disease Area'].dropna().tolist()[1:],
    'Type': dropdown_df['Type'].dropna().tolist(),
    'Model organism': dropdown_df['Model organism'].dropna().tolist(),
    'Biological System (Cell type)': dropdown_df['Cell type'].dropna().tolist(),
    'Assay Design': dropdown_df['Assay format'].dropna().tolist()[1:],
    'Assay readout': dropdown_df['Assay readout'].dropna().tolist()[1:],
    'Screening Mode': dropdown_df['Single Point, Dose Response, Combination'].dropna().tolist()[1:],
    'Data Points': dropdown_df['# of compounds'].dropna().tolist()[1:],
}

for name, vals in dd_cols_of_interest.items():
    print(f'{name}: {vals}')

Disease Area: ['Cancer', 'Neurological ', 'Cardiovascular ', 'Infectious ', 'Inflammatory/immune', 'Metabolic', 'Respiratory ', 'Rare ', 'Gastrointestinal ', 'Hematology']
Type: ['Type', 'Primary screening', 'Hit follow-up', 'Disease profiling', 'Toxicity/Safety models', 'Mechanistic studies', 'Cell phenotyping']
Model organism: ['Model organism', 'Human', 'Mouse', 'Zebrafish', 'Rat', 'Other']
Biological System (Cell type): ['Cell Model', 'Biological System', 'Engineered cell lines', 'Immortalized cell lines', 'Primary cells', 'In vivo', 'Other human material', 'Stem cells/iPSC']
Assay Design: ['2D', '3D', 'Suspension']
Assay readout: ['Luminescence', 'Fluorescence', 'Absorbance', 'Microscopy', 'Flow cytometry', 'other']
Screening Mode: ['Single Point', 'Dose Response', 'Combination']
Data Points: ['<100', '100-1k', '1-5k', '5-20k', '>20k']


## 6. Mapping new Excel → old CSV format

This produces a CSV with the **same column names as the old file**, populated from the new data.
Where the new data has been restructured (e.g. Assay Design split into two columns), we merge them back.

### Cell type: using dropdown-allowed values only

The dropdown sheet defines these allowed values for Cell type (Biological System):
- **Engineered cell lines** — e.g. Cas9 expressing, CRISPR-edited
- **Immortalized cell lines** — e.g. SH-SY5Y, F11, MDCKC, neuron-derived from cell lines
- **Primary cells** — including co-cultures with primary cells
- **In vivo** — animal models
- **Other human material** — e.g. primary antibodies, serum
- **Stem cells/iPSC** — iPSC-derived models

22 rows have NaN in `Biological System`. We infer the correct dropdown value from `Model and modification details`, `Co-culture?`, and `Cell line name(s)`.

In [ ]:
src_temp = new_df.loc[new_df['Type'].notna()].copy()

print(f"=== Biological System value counts (including NaN) ===")
print(src_temp['Biological System'].value_counts(dropna=False).to_string())

print(f"\n=== Rows with NaN Biological System ===")
nan_rows = src_temp[src_temp['Biological System'].isna()]
print(f"Count: {len(nan_rows)}")
print("\nThese rows have details in other columns that help infer the dropdown value:")
for _, r in nan_rows.iterrows():
    md = r['Model and modification details (free text)']
    cc = r['Co-culture?']
    cl = r['Cell line name(s) (free text)']
    print(f"  Model details: {md if pd.notna(md) else '-'} | Co-culture: {cc if pd.notna(cc) else '-'} | Cell line: {cl if pd.notna(cl) else '-'}")

In [ ]:
src = new_df.loc[new_df['Type'].notna()].copy()

src['Assay format'] = src.apply(
    lambda row: (
        f"{row['Assay Design (1)']}, {row['Assay Design (2)']}"
        if pd.notna(row['Assay Design (1)']) and pd.notna(row['Assay Design (2)'])
        else row['Assay Design (1)']
    ),
    axis=1,
)

def merge_readouts(row):
    parts = []
    for col in ['Assay readout', 'Assay readout (2)', 'Assay readout (3)']:
        if pd.notna(row[col]):
            parts.append(str(row[col]).strip())
    return '; '.join(parts) if parts else None

src['Assay readout (merged)'] = src.apply(merge_readouts, axis=1)

screening_mode_map = {
    'Single Point': 'SP',
    'Dose Response': 'DR',
    'Combination': 'DR + combo',
}
src['Screening Mode (mapped)'] = src['Screening Mode'].map(screening_mode_map).fillna(src['Screening Mode'])

def build_cell_type_dropdown(row):
    bio_sys = row['Biological System']
    model_details = row['Model and modification details (free text)']
    co_culture = row['Co-culture?']
    cell_line = row['Cell line name(s) (free text)']

    if pd.notna(bio_sys):
        return str(bio_sys).strip()

    details = str(model_details).strip().lower() if pd.notna(model_details) else ''

    if not details and pd.isna(cell_line):
        return None

    if pd.notna(co_culture) and str(co_culture).strip().lower() == 'yes':
        return 'Primary cells'

    if 'ipsc' in details or 'stem cell' in details:
        return 'Stem cells/iPSC'

    if 'cell line' in details or 'immortalized' in details:
        return 'Immortalized cell lines'

    if 'cas9' in details or 'crispr' in details or 'knock' in details:
        return 'Engineered cell lines'

    if 'neuron' in details:
        return 'Immortalized cell lines'

    if 'neurosphere' in details:
        return 'Immortalized cell lines'

    if pd.notna(cell_line):
        cl = str(cell_line).strip()
        known_lines = ['SH-SY5Y', 'F11', 'MDCKC', 'U2-OS', 'Panc1']
        if any(k in cl for k in known_lines):
            return 'Immortalized cell lines'

    return None

src['Cell type (dropdown)'] = src.apply(build_cell_type_dropdown, axis=1)

print('=== Cell type using dropdown-allowed values ===')
print(src['Cell type (dropdown)'].value_counts(dropna=False).to_string())
print(f'\nTotal non-null: {src["Cell type (dropdown)"].notna().sum()} / {len(src)}')

Merged columns created successfully.
Rows: 54


In [ ]:
output = pd.DataFrame({
    'Site': src['Partner Site Name'],
    'Disease Area': src['Primary Disease Area'],
    'Type': src['Type'],
    'Publication information': src['Publication Details (doi only)'],
    'Model organism': src['Model Organism'],
    'Cell type': src['Cell type (dropdown)'],
    'Cell Type Specifics': src['Model and modification details (free text)'],
    'Assay format': src['Assay format'],
    'Assay readout': src['Assay readout (merged)'],
    'Additional assay information': src['Additional assay information (free text)'],
    '# of compounds': src['Data Points'],
    'Single Point or Dose Response': src['Screening Mode (mapped)'],
    'Additional Notes': src['Additional Details (timepoint, points DRC) \n(free text)'],
    
    'Partner Site Country': src['Partner Site Country'],
    'Tissue of origin': src['Tissue of origin (free text)'],
    'Disease/indication': src['Disease/indication (free text)'],
    'Co-culture?': src['Co-culture?'],
    'Cell line name(s)': src['Cell line name(s) (free text)'],
    'Plate density': src['Plate density'],
    'if 3D: assay type': src['if 3D: assay type'],
    'Measurement Type': src['Measurement Type'],
    'Available to the User': src['Available to the User'],
    'Comments regarding the availability': src['Comments regarding the availability'],
})

print(f'Output shape: {output.shape}')
print(f'\nColumns ({len(output.columns)}):')
for i, c in enumerate(output.columns, 1):
    print(f'  {i:2d}. {c}')

print(f'\n=== Cell type values (dropdown-based) ===')
print(output['Cell type'].value_counts(dropna=False).to_string())

Output shape: (54, 23)

Columns (23):
   1. Site
   2. Disease Area
   3. Type
   4. Publication information
   5. Model organism
   6. Cell type
   7. Cell Type Specifics
   8. Assay format
   9. Assay readout
  10. Additional assay information
  11. # of compounds
  12. Single Point or Dose Response
  13. Additional Notes
  14. Partner Site Country
  15. Tissue of origin
  16. Disease/indication
  17. Co-culture?
  18. Cell line name(s)
  19. Plate density
  20. if 3D: assay type
  21. Measurement Type
  22. Available to the User
  23. Comments regarding the availability


In [9]:
output.head(10)

,Site,Disease Area,Type,Publication information,Model organism,Cell type,Cell Type Specifics,Assay format,Assay readout,Additional assay information,...,Partner Site Country,Tissue of origin,Disease/indication,Co-culture?,Cell line name(s),Plate density,if 3D: assay type,Measurement Type,Available to the User,Comments regarding the availability
0,MEDINA,Cancer,Primary screening,NaN,Human,NaN,Cas9 expressing cell lines,2D,None,NaN,...,ES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MEDINA,Cancer,Primary screening,NaN,Human,NaN,Cas9 expressing cell lines,3D,None,NaN,...,ES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MEDINA,Cancer,Primary screening,10.1016/j.biopha.2024.117018; 10.3390/molecule...,Human,NaN,NaN,3D,None,NaN,...,ES,NaN,NaN,NaN,NaN,NaN,Spheroid/Scaffold-free,NaN,NaN,NaN
3,MEDINA,Cancer,Primary screening,NaN,Human,NaN,NaN,3D,None,NaN,...,ES,Breast,Triple negative breast cancer,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MEDINA,Cancer,Primary screening,10.1371/journal.pone.0167491; 10.1007/978-1-49...,Human,NaN,NaN,2D,None,NaN,...,ES,Bone,Osteosarcoma,NaN,U2-OS,NaN,NaN,NaN,NaN,NaN
5,MEDINA,Neurological,Primary screening,NaN,Human,NaN,neurons derived from human cell line,2D,None,NaN,...,ES,NaN,NaN,NaN,SH-SY5Y,NaN,NaN,NaN,NaN,NaN
6,MEDINA,Neurological,Hit follow-up,NaN,Human,NaN,"dopaminergic neurons, CRISPR-edited LRRK2-ki",2D,None,NaN,...,ES,NaN,NaN,NaN,SH-SY5Y,NaN,NaN,NaN,NaN,NaN
7,MEDINA,Neurological,Hit follow-up,NaN,Human,Stem cells/iPSC,"dopaminergic neurons, CRISPR-edited LRRK2-ki",2D,None,NaN,...,ES,NaN,NaN,NaN,SH-SY5Y,NaN,NaN,NaN,NaN,NaN
8,USC,Neurological,Primary screening,10.1177/2472555218810323,Human,NaN,"dorsal, root ganglia neuron, immortalized F11 ...",NaN,Fluorescence; Microscopy; other,Other: Multi-Electrode Array,...,ES,NaN,NaN,NaN,F11,384-well,NaN,NaN,NaN,NaN
9,USC,Neurological,Primary screening,NaN,Human,NaN,"glutamatergic/dopaminergic neurons, SH-SY5Y ce...",3D,Fluorescence; Microscopy; other,Other: Multi-Electrode Array,...,ES,NaN,NaN,NaN,SH-SY5Y,384-well,NaN,NaN,NaN,NaN


## 7. Validation: check old columns are populated

In [10]:
old_cols = list(old_df.columns)
print('Fill rates for the old-format columns in the new output:\n')
for col in old_cols:
    filled = output[col].notna().sum()
    total = len(output)
    pct = filled / total * 100
    print(f'  {col:40s}  {filled:3d}/{total}  ({pct:5.1f}%)')

Fill rates for the old-format columns in the new output:

  Site                                       54/54  (100.0%)
  Disease Area                               45/54  ( 83.3%)
  Type                                       54/54  (100.0%)
  Publication information                    24/54  ( 44.4%)
  Model organism                             54/54  (100.0%)
  Cell type                                  32/54  ( 59.3%)
  Cell Type Specifics                        28/54  ( 51.9%)
  Assay format                               50/54  ( 92.6%)
  Assay readout                              40/54  ( 74.1%)
  Additional assay information               43/54  ( 79.6%)
  # of compounds                             10/54  ( 18.5%)
  Single Point or Dose Response              35/54  ( 64.8%)
  Additional Notes                           15/54  ( 27.8%)


In [11]:
print('=== Sites ===')
print(f"Old: {sorted(old_df['Site'].dropna().unique())}")
print(f"New: {sorted(output['Site'].dropna().unique())}")

print(f'\n=== Types ===')
print(f"Old: {sorted(old_df['Type'].dropna().str.strip().unique())}")
print(f"New: {sorted(output['Type'].dropna().str.strip().unique())}")

print(f'\n=== Model organism ===')
print(f"Old: {sorted(old_df['Model organism'].dropna().unique())}")
print(f"New: {sorted(output['Model organism'].dropna().unique())}")

print(f'\n=== Assay format ===')
print(f"Old: {sorted(old_df['Assay format'].dropna().unique())}")
print(f"New: {sorted(output['Assay format'].dropna().unique())}")

=== Sites ===
Old: ['CBCS–KI', 'CIPF', 'Fraunhofer ITMP', 'HZI', 'IBCH PAS', 'MEDINA', 'UC', 'UH-FIMM', 'USC', 'UTU', 'i3S']
New: ['CBCS-KI', 'CIPF', 'Fraunhofer ITMP', 'HZI', 'IBCH PAS', 'MEDINA', 'UH-FIMM', 'USC', 'UTU', 'i3S']

=== Types ===
Old: ['Disease Profiling', 'Hit follow-up', 'Primary screening', 'Toxicity models']
New: ['Disease profiling', 'Hit follow-up', 'Primary screening', 'Toxicity/Safety models']

=== Model organism ===
Old: ['human', 'mouse', 'zebrafish']
New: ['Human', 'Mouse', 'Zebrafish']

=== Assay format ===
Old: ['2D', '2D 384', '2D 384 (morphology and MEA in 96)', '2D, cell culture insert', '2D/3D', '384', '3D', '3D & 2D', '3D / 1536', '3D 12 (aim at 96)', '3D 96', 'Scaffold', 'Suspension']
New: ['2D', '2D, 3D', '3D', '3D, 2D', 'Suspension']


In [12]:
output_path = 'Cataloguing screening_v2.csv'
output.to_csv(output_path, index=False)
print(f'Saved new catalogue CSV to: {output_path}')
print(f'Shape: {output.shape}')
print(f'\nThe first 13 columns match the old CSV format.')
print(f'Additional columns from the new Excel file are appended after.')

Saved new catalogue CSV to: Cataloguing screening_v2.csv
Shape: (54, 23)

The first 13 columns match the old CSV format.
Additional columns from the new Excel file are appended after.


In [13]:
verify = pd.read_csv(output_path)
print(f'Reloaded shape: {verify.shape}')
print(f'\nOld CSV columns present in new file:')
for col in old_df.columns:
    status = 'YES' if col in verify.columns else 'MISSING'
    print(f'  [{status}] {col}')

print(f'\nExtra columns in new file:')
for col in verify.columns:
    if col not in old_df.columns:
        print(f'  [NEW] {col}')

Reloaded shape: (54, 23)

Old CSV columns present in new file:
  [YES] Site
  [YES] Disease Area
  [YES] Type
  [YES] Publication information
  [YES] Model organism
  [YES] Cell type
  [YES] Cell Type Specifics
  [YES] Assay format
  [YES] Assay readout
  [YES] Additional assay information
  [YES] # of compounds
  [YES] Single Point or Dose Response
  [YES] Additional Notes

Extra columns in new file:
  [NEW] Partner Site Country
  [NEW] Tissue of origin
  [NEW] Disease/indication
  [NEW] Co-culture?
  [NEW] Cell line name(s)
  [NEW] Plate density
  [NEW] if 3D: assay type
  [NEW] Measurement Type
  [NEW] Available to the User
  [NEW] Comments regarding the availability


In [15]:
# Comparing the shape of old and new dataframes
print(f'Old CSV shape: {old_df.shape}')
print(f'New Excel shape: {new_df.shape}')

Old CSV shape: (57, 13)
New Excel shape: (56, 26)


In [2]:
import pandas as pd
v2_data = pd.read_csv("/home/operation/testing_dstoolkit/IMPULSE_dashboard/data/Cataloguing screening_v2.csv")

In [7]:
diseases_to_rename = {
        "Infectious ": "Infectious Diseases",
        "Metabolic": "Metabolic Disorders",
        "Neurological ": "Neurological Disorders",
    }
v2_data["Disease Area"] = v2_data["Disease Area"].replace(diseases_to_rename)

In [8]:
set(v2_data["Disease Area"])

{'Cancer',
 'Hematology',
 'Infectious Diseases',
 'Metabolic Disorders',
 'Neurological Disorders',
 nan}

In [10]:
# saving the file
v2_data.to_csv("/home/operation/testing_dstoolkit/IMPULSE_dashboard/data/Cataloguing screening_v3.csv", index=False)